# 12 — Agentic RAG: Letting StoryVerse AI Decide

*Advanced series, part 2 of 3.*

Every version of StoryVerse AI so far has had the same rigid shape: a question comes in, we retrieve, we answer. Always. Watch what that costs us on questions that have nothing to do with retrieval at all.


In [ ]:
import os
import json
import faiss
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")
index = faiss.read_index(os.path.join(DATA_DIR, "storyverse_chunks.faiss"))
with open(os.path.join(DATA_DIR, "storyverse_chunks_docstore.json")) as f:
    doc_store = json.load(f)


def retrieve(query: str, top_k: int = 2, min_score: float = 0.35) -> list[dict]:
    query_vec = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = [{**doc_store[i], "score": float(scores[0][j])} for j, i in enumerate(indices[0])]
    return [r for r in results if r["score"] >= min_score]


def rag_answer(question: str) -> str:
    chunks = retrieve(question)
    if not chunks:
        return "I don't have enough information about this in my knowledge base."
    context = "\n\n".join(c["content"] for c in chunks)
    prompt = f"Answer using ONLY this context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.1, reasoning_effort="low")
    return response.choices[0].message.content


In [ ]:
print("'What is 15% of 240?' forced through retrieval:")
print(" ", rag_answer("What is 15% of 240?"))
print()
print("'Hi, how are you?' forced through retrieval:")
print(" ", rag_answer("Hi, how are you?"))


Neither question has anything to do with StoryVerse's stories, yet `rag_answer` retrieves anyway — wasting an embedding call and a similarity search, and handing the model a "no relevant context" situation for a question that never needed context in the first place. A fixed pipeline can't tell the difference between "this needs retrieval" and "this doesn't." It always does the same thing.


## The fix: stop hardcoding the pipeline, give the model tools

Instead of *we* deciding "always retrieve," we hand the model a menu of **tools** it can choose to call — or not — and let it decide per question. Each tool is a plain description: a name, what it does, and what arguments it takes.


In [ ]:
def search_storyverse(query: str) -> str:
    """The actual Python function behind the tool."""
    chunks = retrieve(query, top_k=2)
    if not chunks:
        return "No relevant information found in the StoryVerse knowledge base."
    return "\n\n".join(c["content"] for c in chunks)


def calculator(expression: str) -> str:
    """Evaluate a Python arithmetic expression, e.g. '0.15 * 240'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_storyverse",
            "description": (
                "Search the StoryVerse knowledge base of movies and stories. Always call this "
                "before answering any question about a specific movie, book, or story plot -- "
                "never answer a plot question from your own memory."
            ),
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "search query"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluate a Python arithmetic expression (e.g. 0.15 * 240) and return the numeric "
                "result. Use this for any math instead of computing it yourself."
            ),
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "a valid Python arithmetic expression, e.g. 0.15 * 240"}},
                "required": ["expression"],
            },
        },
    },
]
TOOL_FUNCTIONS = {"search_storyverse": search_storyverse, "calculator": calculator}


Notice the tool descriptions aren't just labels — they're doing real work. "Always call this before answering... never answer from your own memory" does exactly the job `build_rag_prompt`'s "ONLY use the context" instruction did back in notebook 08: it pushes the model to actually check, instead of trusting what it already thinks it knows. Try deleting those two sentences and asking a plot question again — a capable model will often just answer from memory and skip the tool entirely. Giving a model tools doesn't automatically make it use them well. What you write in the tool's description matters just as much as the tool itself.


## The agent loop

An **agentic** system isn't one call — it's a loop: ask the model what to do, run whatever tool it picked, feed the result back, and ask again, until it stops asking for tools and just answers.


In [ ]:
SYSTEM_PROMPT = (
    "You are StoryVerse AI. Always call search_storyverse before answering any question about a "
    "movie, book, or story plot. Always call calculator for arithmetic. For greetings or small "
    "talk, answer directly with no tool call."
)


def run_agent(question: str, max_steps: int = 5) -> tuple[str, list[dict]]:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0.1,
            reasoning_effort="medium",  # tool-calling needs more headroom than plain Q&A
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content, messages  # model is done -- no more tools requested

        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = TOOL_FUNCTIONS[tc.function.name](**args)
            print(f"  [tool call] {tc.function.name}({args}) -> {result[:70]}...")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "(gave up after max_steps -- see 'Where this breaks' below)", messages


## The routing demo

Same three questions as the opening cell — this time, watch the model decide for itself.


In [ ]:
for question in ["Who is Cooper's daughter in Interstellar?", "What is 15% of 240?", "Hi, how are you?"]:
    print("You:", question)
    answer, _ = run_agent(question)
    print("StoryVerse AI:", answer)
    print()


Three different questions, three different behaviors from the exact same function: a `search_storyverse` call for the plot question, a `calculator` call for the arithmetic, and no tool call at all for the greeting. This is **routing** — the model deciding, per question, which tool (if any) actually applies. Nobody wrote an `if "movie" in question` branch anywhere; the routing logic lives entirely in the tool descriptions and the system prompt.


## Chaining tools: one question, two calls

Real questions sometimes need more than one tool, in sequence — the output of one call feeding into the next.


In [ ]:
answer, trace = run_agent("How many moons are in the sky Arjun finds himself under, and what is that number multiplied by 50?")
print("Final answer:", answer)


Watch the tool-call trace above: the model calls `search_storyverse` to find "two moons," *then* calls `calculator` with `2 * 50` using the number it just retrieved. Nobody wired those two calls together manually — the model read its own tool result and decided what to do next based on it. That chaining is the entire difference between "agentic" and a fixed pipeline: the number of steps isn't decided in advance by the code, it's decided live, by the model, based on what comes back.

Don't be surprised if you see the model call `search_storyverse` two or three times with slightly reworded queries before it commits to an answer — that's a real, honest quirk of a 20B model doing multi-step tool use, not a mistake in this notebook. It's a preview of the next section.


## Where this breaks

Agentic systems trade predictability for flexibility. Here's where that predictability breaks down: a question that needs two *separate* retrievals for two unrelated stories, joined in one sentence.


In [ ]:
answer, trace = run_agent("Tell me who Cooper's daughter is, and separately, tell me who betrayed Shivudu's father.")
print("Final answer:", answer)
print("Total messages in trace:", len(trace))


If that hit `max_steps` without ever answering, you just watched a real, common agent failure: the model keeps re-asking a sub-question it already has an answer to, instead of noticing "I have enough, time to respond." This isn't a bug in `run_agent` — it's the honest, real behavior of a smaller model trying to handle two questions stitched into one. Two things make this safe to run in production:

- **`max_steps` as a hard ceiling** — without it, a confused agent keeps going forever, burning tokens and money on every retry. `run_agent` above always stops, even when the model doesn't cooperate.
- **Split compound questions yourself**, instead of hoping the model splits a multi-part question on its own — ask "who is Cooper's daughter?" and "who betrayed Shivudu's father?" as two separate calls to `run_agent`, and both come back clean.

A bigger, more capable model handles this kind of compound question more gracefully. A rigid pipeline never even attempts it. Agentic RAG sits in between: more capable than the pipeline, less reliable than you'd want without guardrails like `max_steps`.


## The same idea, the LangChain / LangGraph way

LangChain wraps this whole loop — call model, run tool, feed result back, repeat — behind `create_agent`. Tools become plain Python functions with a `@tool` decorator; the docstring *is* the tool description.


In [ ]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent


@tool
def search_storyverse_tool(query: str) -> str:
    """Search the StoryVerse knowledge base of movies and stories. Always call this before
    answering any question about a specific movie, book, or story plot."""
    return search_storyverse(query)


@tool
def calculator_tool(expression: str) -> str:
    """Evaluate a Python arithmetic expression (e.g. 0.15 * 240) and return the numeric result."""
    return calculator(expression)


llm = ChatGroq(model=MODEL, temperature=0.1, reasoning_effort="medium")
agent = create_agent(llm, tools=[search_storyverse_tool, calculator_tool], system_prompt=SYSTEM_PROMPT)

for question in ["Who is Cooper's daughter in Interstellar?", "What is 15% of 240?", "Hi, how are you?"]:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("You:", question)
    print("StoryVerse AI:", result["messages"][-1].content)
    print()


Same routing behavior, same three outcomes -- `create_agent` is running the identical retrieve-or-call-or-answer loop we wrote by hand in `run_agent`, just with LangGraph managing the message list and the step-by-step execution internally.

(`create_react_agent` from `langgraph.prebuilt` does the same thing, but is now deprecated in favor of `create_agent` from `langchain.agents` -- if you see that name in older tutorials, this is its replacement.)


## What makes this "agentic" instead of just "RAG"

| | Plain RAG (notebooks 07-10) | Agentic RAG (this notebook) |
|---|---|---|
| Retrieval | Always happens, every question | Happens only if the model decides it's needed |
| Number of steps | Fixed: retrieve once, answer once | Variable: zero, one, or several tool calls, decided live |
| Tools available | Just the vector store | Anything you give it a description for -- calculators, APIs, other retrievers |
| Failure mode | Wrong or missing chunks | Wrong chunks, *or* looping, *or* calling the wrong tool -- more ways to fail, in exchange for more flexibility |

Agentic RAG isn't strictly "better" than plain RAG — it's the right tool when a question's shape genuinely isn't known in advance. For a system that only ever answers questions about your document library, the fixed pipeline from notebook 08 is simpler, cheaper, and more predictable. Reach for an agent when some questions need retrieval, some need something else entirely, and you don't want to hand-write that branch yourself.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Tool | A described function the model can choose to call, with a name, description, and arguments |
| Tool calling / function calling | The model requesting a specific tool call instead of answering directly |
| Agent loop | Ask the model, run whatever tool it picked, feed the result back, repeat until it stops asking |
| Routing | The model choosing which tool (if any) applies to a given question |
| `max_steps` | A hard ceiling on how many tool-call rounds an agent gets, so a confused loop can't run forever |

**Next up:** notebook 13 gives StoryVerse AI eyes — reading images and PDFs, not just plain text files.

**Exercises:**

- Add a third tool — maybe `get_today_date()` — and ask a question that needs it.
- Lower `max_steps` to 2 and re-run the compound question from "Where this breaks" — does the failure message change?
- Rewrite `SYSTEM_PROMPT` to remove the "always call search_storyverse" instruction entirely, and ask a plot question — does the model still bother retrieving?
